# CNITS - Hameedat Vaccine Tracker Prototype

This notebook is for prototyping and understanding the vaccination logic before moving it into `src/`.
The schedule comes from the NPHCDA immunization schedule supplied by the team.

In [ ]:
from src.vaccine_tracker import calculate_age, build_vaccine_schedule, group_vaccines_by_status

## 1. Calculate the child's age

We use exact elapsed days for week-based milestones and calendar-month arithmetic for month-based milestones.

In [ ]:
calculate_age("2026-01-01", "2026-03-01")

## 2. Build the child's vaccine schedule

For a first prototype, we can assume no vaccine history has been entered. The tracker will therefore mark past milestones as overdue and future milestones as upcoming.

In [ ]:
records = build_vaccine_schedule("2026-01-01", as_of="2026-05-01")
grouped = group_vaccines_by_status(records)
{key: len(value) for key, value in grouped.items()}

In [ ]:
for status, items in grouped.items():
    print(f"\n{status.upper()}")
    for item in items:
        print(item["vaccine"], "|", item["due_date_label"], "|", item["dosage"], "|", item["route"], "|", item["site"])

## 3. Add vaccination history

The history should contain the actual date a dose was administered. This prevents a completed vaccine from being incorrectly displayed as overdue.

In [ ]:
history = {
    "bcg_birth": "2026-01-01",
    "opv0_birth": "2026-01-01",
    "hep_b0_birth": "2026-01-01",
    "pentavalent1_6w": "2026-02-15",
    "pcv1_6w": "2026-02-15",
    "opv1_6w": "2026-02-15",
    "rota1_6w": "2026-02-15",
    "ipv1_6w": "2026-02-15",
}

records_with_history = build_vaccine_schedule("2026-01-01", history, as_of="2026-05-01")
grouped_with_history = group_vaccines_by_status(records_with_history)
{key: len(value) for key, value in grouped_with_history.items()}

## 4. Important project decision: HPV

The source gives HPV as a 9-13 year window and says two doses are separated by 6 months. It does not specify one exact first-dose date. Therefore, CNITS should not invent one. Dose 1 is marked **Due Now** during the age window, while dose 2 is calculated from the recorded dose-1 date.

In [ ]:
hpv_records = build_vaccine_schedule(
    "2018-06-01",
    vaccination_history={"hpv1": "2027-07-15"},
    as_of="2027-12-01",
)
[r for r in hpv_records if r["series"] == "HPV"]

## 5. Calendar reminder

After `ics` is installed in the project environment, call `generate_ics_calendar()` from `vaccine_tracker.py`. It creates an event on the exact due date and attaches a reminder three days before the event.